# Start new work from main
1. Type your branch name in the box at the top.
2. Click **Run all**.
3. Wait for the green box, then open your Git folder as usual.

In [0]:
import os, subprocess
from databricks.sdk import WorkspaceClient

REPO_NAME = "git-test"
DEFAULT_BRANCH = "main"

dbutils.widgets.text("branch", "", "Your branch name")
branch = dbutils.widgets.get("branch").strip()

me = WorkspaceClient().current_user.me().user_name
repo = f"/Workspace/Users/{me}/{REPO_NAME}"

def git(*args):
    r = subprocess.run(["git", "-C", repo, *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"Git step failed ({' '.join(args)}). "
                           f"Please send this to the dev team:<br><pre>{r.stderr}</pre>")
    return r.stdout.strip()

def show(msg, ok=True):
    color = "#e6f4ea" if ok else "#fdecea"
    displayHTML(f"<div style='padding:16px;border-radius:8px;background:{color};"
                f"font-size:16px;font-family:sans-serif'>{msg}</div>")

try:
    if not branch:
        raise RuntimeError("Please type your branch name in the box at the top, then click Run all again.")
    if not os.path.exists(f"{repo}/.git"):
        raise RuntimeError("Your Git folder isn't set up for this tool yet. Please contact the dev team.")
    if git("status", "--porcelain"):
        raise RuntimeError("You have unsaved changes. Commit or discard them in the Git dialog first.")

    git("fetch", "--prune", "origin")

    # Branch still on GitHub = real work in progress: just open it, never overwrite
    if git("ls-remote", "--heads", "origin", branch):
        git("checkout", branch)
        git("pull", "--ff-only", "origin", branch)
        show(f"Your branch <b>{branch}</b> already exists on GitHub, so I opened it for you.")
    else:
        git("checkout", DEFAULT_BRANCH)
        git("pull", "--ff-only", "origin", DEFAULT_BRANCH)

        # Remove local branches whose GitHub copy was deleted
        lines = git("for-each-ref", "--format=%(refname:short)|%(upstream:track)", "refs/heads").splitlines()
        for b in [l.split("|")[0] for l in lines if "[gone]" in l]:
            git("branch", "-D", b)

        # Leftover local-only branch with the same name (never pushed)
        if git("branch", "--list", branch):
            raise RuntimeError(f"There is an unpushed branch called <b>{branch}</b> in your folder. "
                               "Please contact the dev team so nothing is lost.")

        git("checkout", "--no-track", "-b", branch, f"origin/{DEFAULT_BRANCH}")
        show(f"Done! <b>{branch}</b> was created fresh from the latest {DEFAULT_BRANCH}. You can start working.")
except Exception as e:
    show(str(e), ok=False)